# Tutorial: Feature Selection & Dimensionality Reduction

> **Type:** Tutorial. Follow along step by step. Your only job is to make it work. Don't skip cells.
>
> **Time:** ~1.5-2 hours
> **Prerequisites:** Module 14 (Feature Engineering), Module 05 (Information Theory)
> **What you'll build:** A side-by-side comparison of filter, wrapper, and embedded feature selection, plus PCA, on a noisy synthetic dataset. Then a visual comparison of PCA, t-SNE, and UMAP on a real digits dataset.

---

## What We're Building

Two parts. First, we build a synthetic classification dataset deliberately stuffed with redundant and pure-noise features. We run every selection and reduction technique from `explanation.md` against it, and compare downstream model accuracy and training time. Second, we take a real dataset (handwritten digits) and compare how PCA, t-SNE, and UMAP each compress 64 dimensions down to 2 for visualisation. You'll see the local-vs-global structure trade-off for yourself.

Don't worry yet about understanding every line. That's what `explanation.md` is for. Here, just follow the steps and watch it work.

## Step 0: Setup

Install and import everything we need.

In [ ]:
# Install dependencies (uncomment if running for the first time)
# !pip install numpy pandas scikit-learn matplotlib umap-learn

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, load_digits
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, mutual_info_classif,
    RFECV, SelectFromModel,
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False
    print("umap-learn not installed — the UMAP section will be skipped. "
          "Install with `pip install umap-learn` to run it.")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11
print("Setup complete.")

## Step 1: Build a Deliberately Redundant Dataset

We use `make_classification` to build a dataset where we control exactly how much signal exists: 50 features in total, but only 10 are informative. Another 10 are linear combinations of the informative ones (redundant), and the remaining 30 are pure noise. This mirrors a realistic engineered dataset, with lots of columns of which only some matter. It lets us check whether each selection method finds the useful features.

In [ ]:
X, y = make_classification(
    n_samples=2000,
    n_features=50,
    n_informative=10,
    n_redundant=10,
    n_repeated=0,
    n_clusters_per_class=2,
    flip_y=0.02,
    random_state=RANDOM_SEED,
)
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print("True composition: 10 informative, 10 redundant, 30 pure noise features")

> **Check:** You should see a train shape of `(1500, 50)` and a test shape of `(500, 50)`.

## Step 2: Baseline — All 50 Features

Before reducing anything, establish the baseline: accuracy and training time using every feature. Everything else in this notebook gets compared back to this.

In [ ]:
def evaluate(X_tr, X_te, y_tr, y_te, label, model=None):
    """Fit a RandomForest, time it, and report test accuracy."""
    if model is None:
        model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED)
    start = time.time()
    model.fit(X_tr, y_tr)
    fit_time = time.time() - start
    acc = model.score(X_te, y_te)
    print(f"{label:32s} | n_features={X_tr.shape[1]:3d} | fit_time={fit_time:.3f}s | test_accuracy={acc:.4f}")
    return {"label": label, "n_features": X_tr.shape[1], "fit_time": fit_time, "accuracy": acc}

results = []
results.append(evaluate(X_train, X_test, y_train, y_test, "Baseline (all 50 features)"))

## Step 3: Filter Selection — Variance Threshold + Mutual Information

Filter methods score each feature independently of any model. We first drop near-zero-variance columns. None exist here, because `make_classification` produces reasonably scaled features, but this is always the first pass on real data. Then we rank by mutual information with the target and keep the top 15.

In [ ]:
# Variance threshold (mostly a no-op here, but always run it first on real data)
var_selector = VarianceThreshold(threshold=0.0)
var_selector.fit(X_train)
print(f"Features dropped by variance threshold: {(~var_selector.get_support()).sum()}")

# Mutual information: keep top 15
mi_selector = SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, random_state=RANDOM_SEED), k=15)
X_train_mi = mi_selector.fit_transform(X_train, y_train)
X_test_mi = mi_selector.transform(X_test)
mi_selected_features = X_train.columns[mi_selector.get_support()].tolist()

print(f"\nTop 15 features by mutual information:\n{mi_selected_features}")
results.append(evaluate(X_train_mi, X_test_mi, y_train, y_test, "Filter: mutual information (top 15)"))

> **What just happened?** Mutual information scored all 50 features against the label and kept the 15 most informative. We know the ground truth (10 informative, 10 redundant, 30 noise), but `make_classification` shuffles the columns by default, so the useful features aren't in any particular position. To check the filter against the truth, rebuild the data with `shuffle=False`, which puts the informative and redundant features in `feature_0` to `feature_19`.

## Step 4: Wrapper Selection — RFECV

RFECV repeatedly trains a model, drops the weakest feature(s), and uses cross-validation to find the feature count that maximises validation performance. You don't need to guess `k` in advance. It's slower, because it retrains many times, but it directly optimises what we care about.

In [ ]:
rfe_estimator = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
rfecv = RFECV(estimator=rfe_estimator, step=2, cv=5, scoring='accuracy', n_jobs=-1)

start = time.time()
rfecv.fit(X_train, y_train)
rfe_fit_time = time.time() - start

print(f"RFECV selected {rfecv.n_features_} features in {rfe_fit_time:.1f}s")
rfe_selected_features = X_train.columns[rfecv.support_].tolist()

X_train_rfe = X_train[rfe_selected_features]
X_test_rfe = X_test[rfe_selected_features]
results.append(evaluate(X_train_rfe, X_test_rfe, y_train, y_test, f"Wrapper: RFECV ({rfecv.n_features_} features)"))

## Step 5: Embedded Selection — Random Forest Importances

Train one Random Forest, read off its feature importances, and keep everything above the median. This is the cheapest method that still uses the model's own behaviour to decide what matters.

In [ ]:
rf_for_importance = RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED)
rf_for_importance.fit(X_train, y_train)

importances = pd.Series(rf_for_importance.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 10 features by Random Forest importance:")
print(importances.head(10))

embedded_selector = SelectFromModel(rf_for_importance, prefit=True, threshold='median')
X_train_embedded = embedded_selector.transform(X_train)
X_test_embedded = embedded_selector.transform(X_test)

results.append(evaluate(X_train_embedded, X_test_embedded, y_train, y_test,
                         f"Embedded: RF importance (median threshold)"))

## Step 6: Feature Extraction — PCA

Now switch from *selection* to *extraction*. Scale the data, fit PCA, and see how many components are needed to capture 95% of the variance. Only 20 features carry signal, and 10 of those are linear combinations of the other 10. But after scaling, each of the 30 noise features carries as much variance as a useful one, so watch how that affects the component count.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca_full = PCA(random_state=RANDOM_SEED).fit(X_train_scaled)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(range(1, 21), pca_full.explained_variance_ratio_[:20])
axes[0].set_xlabel('Principal component')
axes[0].set_ylabel('Explained variance ratio')
axes[0].set_title('Scree plot (first 20 components)')

axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', markersize=3)
axes[1].axhline(0.95, color='red', linestyle='--', label='95% variance')
axes[1].set_xlabel('Number of components')
axes[1].set_ylabel('Cumulative explained variance')
axes[1].set_title('Cumulative explained variance')
axes[1].legend()
plt.tight_layout()
plt.show()

n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f"Components needed for 95% variance: {n_components_95} (out of 50 original features)")

In [ ]:
pca = PCA(n_components=n_components_95, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

results.append(evaluate(X_train_pca, X_test_pca, y_train, y_test,
                         f"Extraction: PCA ({n_components_95} components)"))

> **What just happened?** PCA needed 37 of the 50 components to reach 95% of the variance, not a handful. The redundant features did collapse onto the informative ones, but after standardisation every noise feature has unit variance too. PCA keeps directions of high *variance*, not high *relevance*, so it can't tell signal from noise when both have the same spread. That's the key limit of PCA as a feature-reduction step: it's unsupervised.

## Step 7: Compare Everything

Line up every method's feature count, training time, and test accuracy side by side.

In [ ]:
results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False).reset_index(drop=True)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4E79A7' if 'Baseline' not in label else '#E15759' for label in results_df['label']]
ax.barh(results_df['label'], results_df['accuracy'], color=colors)
ax.set_xlabel('Test accuracy')
ax.set_xlim(min(results_df['accuracy']) - 0.02, 1.0)
ax.set_title('Accuracy by feature selection / extraction method')
plt.tight_layout()
plt.show()

> **What just happened?** The two methods that use the model's own behaviour came out ahead. RFECV (26 features, 0.928) and Random Forest importance (25 features, 0.922) both beat the all-features baseline (0.912) with about half the features, because dropping noise reduces overfitting. The other two did worse. The mutual-information filter kept only 15 features and lost accuracy (0.884), because it scores each feature alone and the cut-off was too tight. PCA (37 components, 0.868) did worst, for the reason in Step 6: it compresses variance, not signal. Fewer features can mean better accuracy, but only when the method that picks them knows what the model needs.

## Step 8: Visual Comparison — PCA vs. t-SNE vs. UMAP

Selection and PCA are about *modelling*. Now switch to the other use case: pure visualisation. We load the classic digits dataset (8x8 pixel images of handwritten digits 0-9, with 64 pixel-intensity features) and compare how PCA, t-SNE, and UMAP each compress it to 2D.

In [ ]:
digits = load_digits()
X_digits, y_digits = digits.data, digits.target
X_digits_scaled = StandardScaler().fit_transform(X_digits)

print(f"Digits dataset: {X_digits.shape[0]} samples, {X_digits.shape[1]} pixel features, {len(np.unique(y_digits))} classes")

In [ ]:
# PCA: fast, deterministic, linear
start = time.time()
X_digits_pca = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X_digits_scaled)
print(f"PCA: {time.time() - start:.2f}s")

# t-SNE: slower, non-linear, local structure
start = time.time()
X_digits_tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_SEED, init='pca').fit_transform(X_digits_scaled)
print(f"t-SNE: {time.time() - start:.2f}s")

# UMAP: non-linear, usually faster than t-SNE, if installed
if HAVE_UMAP:
    start = time.time()
    X_digits_umap = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=RANDOM_SEED).fit_transform(X_digits_scaled)
    print(f"UMAP: {time.time() - start:.2f}s")

In [ ]:
n_plots = 3 if HAVE_UMAP else 2
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5.5))

embeddings = [(X_digits_pca, 'PCA'), (X_digits_tsne, 't-SNE')]
if HAVE_UMAP:
    embeddings.append((X_digits_umap, 'UMAP'))

for ax, (embedding, title) in zip(axes, embeddings):
    scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')

fig.colorbar(scatter, ax=axes, label='Digit class', fraction=0.02, pad=0.02)
plt.suptitle('Same 64-dimensional digit data, three 2D embeddings', y=1.02)
plt.show()

> **What just happened?** PCA gives a reasonable but overlapping view, because it optimises for global variance, not class separation. t-SNE and UMAP preserve local neighbourhoods, so they usually show much tighter, better-separated clusters per digit, even though neither was told the labels. Notice the timing, too. PCA is near-instant. On this run, UMAP (8.6s) was slower than t-SNE (1.5s); on a dataset this small, UMAP's setup cost dominates, while on large datasets it's usually the faster of the two. UMAP also tends to keep more of the global layout than t-SNE.

## What You Built

You ran every major feature reduction technique (variance and mutual-information filtering, RFECV, embedded Random Forest importance, and PCA extraction) against a dataset with known noise. The model-aware methods found smaller feature sets that beat the baseline. The filter and PCA showed their limits. You also produced a direct visual comparison of PCA, t-SNE, and UMAP on real image data, the picture to keep in mind whenever you reach for one of these three.

## What's Next

- Read `explanation.md` if you haven't, to connect what you just ran to the theory (the curse of dimensionality, why PCA is an eigendecomposition, why t-SNE and UMAP output shouldn't feed a model).
- Open `howto.md` when you need to repeat any of these steps on your own data.
- Keep `reference.md` open as a cheat sheet for API signatures and the decision guide.
- **Extension challenge:** rerun Step 7 with `n_redundant=0` (pure noise instead of correlated redundancy) and see how the comparison changes. Filter methods should hold up, but PCA has even less correlated structure to compress.

---

**AWS SageMaker version:** for datasets too large to fit in memory, replace the `sklearn.decomposition.PCA` step with SageMaker's built-in distributed `PCA` estimator. See `howto.md` for the equivalent call.